In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

path = "data/Customer-Churn-Records.csv"

df = pd.read_csv(path)
df.head()



,RowNumber,CustomerId,Surname,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited,Complain,Satisfaction Score,Card Type,Point Earned
0,1,15634602,Hargrave,619,France,Female,42,2,0.00,1,1,1,101348.88,1,1,2,DIAMOND,464
1,2,15647311,Hill,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0,1,3,DIAMOND,456
2,3,15619304,Onio,502,France,Female,42,8,159660.80,3,1,0,113931.57,1,1,3,DIAMOND,377
3,4,15701354,Boni,699,France,Female,39,1,0.00,2,0,0,93826.63,0,0,5,GOLD,350
4,5,15737888,Mitchell,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0,0,5,GOLD,425


In [3]:
# Data profiling and validation before EDA

# 1) Basic shape and quality checks
print('Shape:', df.shape)
print('\nMissing values:')
print(df.isna().sum())
print('\nDuplicate rows:', df.duplicated().sum())



Shape: (10000, 18)

Missing values:
RowNumber             0
CustomerId            0
Surname               0
CreditScore           0
Geography             0
Gender                0
Age                   0
Tenure                0
Balance               0
NumOfProducts         0
HasCrCard             0
IsActiveMember        0
EstimatedSalary       0
Exited                0
Complain              0
Satisfaction Score    0
Card Type             0
Point Earned          0
dtype: int64

Duplicate rows: 0


In [4]:
# 2) Data types and basic summaries
print('\nDtypes:')
print(df.dtypes)
print('\nNumeric summary:')
print(df.describe(include='number').T)
print('\nCategorical summary:')
for col in df.select_dtypes(exclude='number').columns:
    print(f'[{col}]')
    print(df[col].value_counts(dropna=False).head(10))



Dtypes:
RowNumber               int64
CustomerId              int64
Surname                   str
CreditScore             int64
Geography                 str
Gender                    str
Age                     int64
Tenure                  int64
Balance               float64
NumOfProducts           int64
HasCrCard               int64
IsActiveMember          int64
EstimatedSalary       float64
Exited                  int64
Complain                int64
Satisfaction Score      int64
Card Type                 str
Point Earned            int64
dtype: object

Numeric summary:
                      count          mean           std          min  \
RowNumber           10000.0  5.000500e+03   2886.895680         1.00   
CustomerId          10000.0  1.569094e+07  71936.186123  15565701.00   
CreditScore         10000.0  6.505288e+02     96.653299       350.00   
Age                 10000.0  3.892180e+01     10.487806        18.00   
Tenure              10000.0  5.012800e+00      2.892174    

In [5]:
# 3) Rule-based sanity checks
print('\nSanity checks:')
print('CreditScore range check:', ((df['CreditScore'] < 300) | (df['CreditScore'] > 850)).sum(), 'rows outside expected range')
print('Age range check:', ((df['Age'] < 18) | (df['Age'] > 100)).sum(), 'rows outside expected range')
print('Tenure range check:', ((df['Tenure'] < 0) | (df['Tenure'] > 10)).sum(), 'rows outside expected range')
print('Balance negative check:', (df['Balance'] < 0).sum(), 'rows')
print('NumOfProducts range check:', ((df['NumOfProducts'] < 1) | (df['NumOfProducts'] > 4)).sum(), 'rows outside expected range')
print('Exited values check:', sorted(df['Exited'].unique()), 'expected [0, 1]')
print('HasCrCard values check:', sorted(df['HasCrCard'].unique()), 'expected [0, 1]')
print('IsActiveMember values check:', sorted(df['IsActiveMember'].unique()), 'expected [0, 1]')
print('Complain values check:', sorted(df['Complain'].unique()), 'expected [0, 1]')
print('Satisfaction Score range check:', ((df['Satisfaction Score'] < 1) | (df['Satisfaction Score'] > 5)).sum(), 'rows outside expected range')





Sanity checks:
CreditScore range check: 0 rows outside expected range
Age range check: 0 rows outside expected range
Tenure range check: 0 rows outside expected range
Balance negative check: 0 rows
NumOfProducts range check: 0 rows outside expected range
Exited values check: [np.int64(0), np.int64(1)] expected [0, 1]
HasCrCard values check: [np.int64(0), np.int64(1)] expected [0, 1]
IsActiveMember values check: [np.int64(0), np.int64(1)] expected [0, 1]
Complain values check: [np.int64(0), np.int64(1)] expected [0, 1]
Satisfaction Score range check: 0 rows outside expected range


In [6]:
# 4) Columns to drop before EDA
drop_cols = ['RowNumber', 'Surname']
print('\nColumns to drop before EDA:', drop_cols)

eda_df = df.drop(columns=drop_cols, errors='ignore')
eda_df.head()


Columns to drop before EDA: ['RowNumber', 'Surname']


,CustomerId,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited,Complain,Satisfaction Score,Card Type,Point Earned
0,15634602,619,France,Female,42,2,0.00,1,1,1,101348.88,1,1,2,DIAMOND,464
1,15647311,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0,1,3,DIAMOND,456
2,15619304,502,France,Female,42,8,159660.80,3,1,0,113931.57,1,1,3,DIAMOND,377
3,15701354,699,France,Female,39,1,0.00,2,0,0,93826.63,0,0,5,GOLD,350
4,15737888,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0,0,5,GOLD,425


In [7]:
# 5) Final rename and export for silver layer
rename_cols = {
    'Satisfaction Score': 'SatisfactionScore',
    'Card Type': 'CardType',
    'Point Earned': 'PointEarned'
}

silver_df = eda_df.rename(columns=rename_cols)
print('Renamed columns:')
print([rename_cols[col] for col in rename_cols])

import os
os.makedirs('data/silver', exist_ok=True)

silver_csv_path = 'data/silver/customer_churn_silver.csv'
silver_df.to_csv(silver_csv_path, index=False)
print('Saved silver CSV:', silver_csv_path)

silver_parquet_path = 'data/silver/customer_churn_silver.parquet'
try:
    silver_df.to_parquet(silver_parquet_path, index=False)
    print('Saved silver Parquet:', silver_parquet_path)
except Exception as exc:
    print('Parquet export skipped or failed:', exc)
    print('CSV output is still available.')

silver_df.head()

Renamed columns:
['SatisfactionScore', 'CardType', 'PointEarned']
Saved silver CSV: data/silver/customer_churn_silver.csv
Saved silver Parquet: data/silver/customer_churn_silver.parquet


,CustomerId,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited,Complain,SatisfactionScore,CardType,PointEarned
0,15634602,619,France,Female,42,2,0.00,1,1,1,101348.88,1,1,2,DIAMOND,464
1,15647311,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0,1,3,DIAMOND,456
2,15619304,502,France,Female,42,8,159660.80,3,1,0,113931.57,1,1,3,DIAMOND,377
3,15701354,699,France,Female,39,1,0.00,2,0,0,93826.63,0,0,5,GOLD,350
4,15737888,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0,0,5,GOLD,425
